## Original SCB agent prompt provided in the paper
### Before the modular impl. agents are implemented, use this 
Specify additionally that follow the design in `current_design.json`

In [4]:
from prompts.scb import get_original_scb_prompt

# Pass the current checkpoint number to be implemented 
print(get_original_scb_prompt(checkpoint_number=2))


Implement a program that 100% solves the specification.
That is all you need to do.

Keep using the same virtual environment you started with,
update 'requirements.txt' with any new dependencies you need.

You are working on the following issue: checkpoint_2.md
Implement your solution in implementation/ folder.
Extend your solution based on previous_implementation/.



## Reader agent (Unnecessary because we have metrics?)
Its purpose is to intiialise the `current_design.json` for the loop, and does not output anything. 

In [ ]:
from prompts.reader import get_reader_prompt

print(get_reader_prompt(3))

## Analyzer agent

In [6]:
from prompts.analyzer import get_analyzer_prompt

# The actual DPy results should be filtered for specific things we concern only. 
# For this TEST example, a path is provided instead. DO NOT do this in your final submission 
print(get_analyzer_prompt(has_implementation=False))


You are a senior software code quality analyst.

Your job is to evaluate the following modular design including kept, changed or new modules.
Project root: agent_workspace
Design: `current_design.json`
Dependency graph: `current_deps_graph.json`

Your evaluation is based on the following master criteria to achieve long term code quality and maintainability: 
- Modules should be able to independently evolve, each having only a single responsibility separated by clear boundaries. 
- The system should be easy to test by avoiding overly complex functions with lots of control paths that could be simplified if possible.
- The maintenance effort when introducing new features should be as low as possible, particularly for modules that are likely to change. Avoid tight coupling by minimising duplication and directly accessing private elements. 

Give analyzer improvement suggestions based on the following: 
- If the module has output of different abstractions, or a method has too many postcond

## Decomposer Agent

In [5]:
from prompts.decomposer import get_decomposer_prompt

# Extract only the improvements part of the analyzer output 
print(get_decomposer_prompt(1))


You are a senior software engineer that specialises in modular software design.

You are working on the following issue:
Project root: agent_workspace
Issue path: checkpoint_1.md


If a design is provided in `current_design.json` with the dependency graph in `current_deps_graph.json`, prioritise reusing existing modules instead of creating a new module where possible.
If a list of improvement suggestions for the current design is provided in `current_analyzer_result.json`, please consider accepting or rejecting them based on: 
- Whether the suggestion lead to reduced future effort when adding new features.
- Whether the nature of the problem justifies the current complexity without the refactoring.
- Whether the suggestion conflict with issue requirements.

Propose a modular design that achieves the goal specified in the issue when integrated together. 
You should follow best code practices, including: 
- A module should only expose the minimum amount of knowledge in its public interf

## Analyzer on the implementation

In [10]:
from prompts.analyzer import get_analyzer_prompt

print(get_analyzer_prompt(has_implementation=True))


You are a senior software code quality analyst.

Your job is to evaluate the following implementation including kept, changed or new modules.
Project root: agent_workspace
Implementation: `implementation/
Dependency graph: `current_deps_graph.json`

Your evaluation is based on the following master criteria to achieve long term code quality and maintainability: 
- Modules should be able to independently evolve, each having only a single responsibility separated by clear boundaries. 
- The system should be easy to test by avoiding overly complex functions with lots of control paths that could be simplified if possible.
- The maintenance effort when introducing new features should be as low as possible, particularly for modules that are likely to change. Avoid tight coupling by minimising duplication and directly accessing private elements. 

Give analyzer improvement suggestions based on the following: 
- If the module has output of different abstractions, or a method has too many postc

In [ ]:
ANALYZER_OUTPUT_2 = {
    "result": "pass",
    "improvements": [
      {
        "module_name": "pipeline.ast_nodes",
        "smell": "Module conflates language-level AST node types with application-level caching configuration structures, reducing cohesion and coupling the language frontend to the caching subsystem.",
        "improvement_instruction": "Extract TtlConfig, CacheKeyConfig, CacheConfig, and GlobalCacheConfig into a dedicated pipeline.cache_config module. pipeline.ast_nodes should retain only constructs that represent parsed language elements (TaskDef, ParamDef, and token-adjacent types). Update imports in pipeline.cache_key, pipeline.cache_manager, pipeline.executor, and pipeline.main accordingly."
      },
      {
        "module_name": "pipeline.expr_parser",
        "smell": "parse_block returns List[Any], erasing all type information at the parser/evaluator boundary despite typed AST node dataclasses existing in pipeline.ast_nodes.",
        "improvement_instruction": "Define a StmtNode union type or a common base dataclass in pipeline.ast_nodes that covers all statement node variants (IfStmt, ForStmt, WhileStmt, AssignStmt, ReturnStmt, etc.). Change ExprParser.parse_block to return List[StmtNode] so that static type checking is preserved across the parse/evaluate boundary."
      },
      {
        "module_name": "pipeline.evaluator",
        "smell": "eval_block accepts raw List[Token] and internally invokes ExprParser, conflating token parsing with expression evaluation and violating the established lex-parse-evaluate layering.",
        "improvement_instruction": "Remove the token-to-AST parsing step from eval_block. Change its signature to accept List[StmtNode] (a pre-parsed AST). Callers such as Executor should invoke ExprParser.parse_block explicitly before calling eval_block, keeping the two phases separately testable and aligned with the pipeline.expr_parser/pipeline.evaluator module boundary."
      },
      {
        "module_name": "pipeline.cache_manager",
        "smell": "store() accepts both the precomputed cache_key and the raw inputs (task_def, params, workspace) from which the key was derived, producing a redundant and inconsistent method signature.",
        "improvement_instruction": "Simplify store() to accept only task_def (for cache location resolution), cache_key, and job_result. Remove the redundant params and workspace parameters; since cache_key is already computed by a prior check() call, only the cache directory (derivable from task_def.cache.location) is needed to persist the entry."
      },
      {
        "module_name": "pipeline.cache_store",
        "smell": "The exists() method is fully subsumed by load() returning None, unnecessarily widening the public interface and enabling TOCTOU access patterns.",
        "improvement_instruction": "Remove the exists() method from cache_store's public interface. Update all callers in pipeline.cache_manager to use load() and branch on the None return value, eliminating the separate existence check."
      }
    ]
  }

# Modular coder agent

In [8]:
from prompts.modular_coder import get_modular_coder_prompt

print(
    get_modular_coder_prompt(1,['pipeline/requires_executor.py', 'pipeline/success_evaluator.py'])
)


You are working on the following issue: checkpoint_1.md
Implement your solution in implementation/ folder.


Use a virtual environment and ensure that a 'requirements.txt' is present with any dependencies
you need to solve the problem.

Ensure your code a good quality, below are a list of practices you should follow but not limited to them. 
- Avoids functions that are too complex with too much nested if/else statements.
- Avoid the use of magic numbers when their meanings are not obvious.
- Do not access the private elements of another class.


Your job is to implement parts of the design specified in `current_design.json`, and you should import existing modules. 
You should ONLY implement the following 2 modules: 
- pipeline/requires_executor.py
- pipeline/success_evaluator.py


